# 🚀 Модуль 11. Урок 4. Text-to-SQL RAG-ассистент: индексация схемы DDL и метаданных витрин

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_11_rag_vector_db/lesson_04_text_to_sql_rag_assistant.ipynb)

## 📌 Бизнес-контекст и методология
Если в корпоративном хранилище 500 таблиц, передать всю схему в промпт невозможно (превышение контекста и деградация внимания).
Решение: RAG над схемой данных. Мы индексируем описания таблиц и колонок, извлекаем только топ-3 релевантные таблицы и передаем их DDL в LLM.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
table_catalog = {
    "dim_customers": "Справочник клиентов, города, каналы привлечения и дата регистрации",
    "fct_orders": "Транзакции заказов, статус оплаты, дата совершения покупки",
    "fct_server_logs": "Технические логи latency микросервисов и ошибок 500"
}

user_biz_question = "Сколько новых клиентов зарегистрировалось из Москвы?"

# Простой эвристический семантический матчер для выбора схемы
selected_tables = []
for tbl, desc in table_catalog.items():
    if "клиент" in desc.lower() or "регистраци" in desc.lower():
        selected_tables.append(tbl)

print(f"Бизнес-вопрос: {user_biz_question}")
print(f"Автоматически отобранные релевантные таблицы схемы: {selected_tables}")


Бизнес-вопрос: Сколько новых клиентов зарегистрировалось из Москвы?
Автоматически отобранные релевантные таблицы схемы: ['dim_customers']


### 🎯 Практическое задание 1: Валидация отбора таблиц
Убедитесь, что таблица `dim_customers` выбрана, а `fct_server_logs` исключена.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert "dim_customers" in selected_tables
assert "fct_server_logs" not in selected_tables
print("✅ Тест успешно пройден: редукция схемы выполнена успешно!")


✅ Тест успешно пройден: редукция схемы выполнена успешно!


### 💼 Архитектура Enterprise Text-to-SQL
1. Vector Search по метаданным таблиц -> 2. Сборка DDL топ-5 таблиц -> 3. Генерация SQL -> 4. Dry-run валидация через EXPLAIN.